# ByteResearch on Colab
Run the cells from top to bottom on a GPU runtime. Every output (data, checkpoints, results, logs, filter
tables) is written to Google Drive, so re-running a cell skips finished units and resumes interrupted ones.
Only the COMET/MetricX venv lives on the local disk (`/content/envs`).

In [ ]:
# Google Drive first: all artifacts go there, and nothing runs unless Drive is mounted.
import os
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive"
ARTIFACTS = f"{DRIVE}/MyDrive/ByteResearch/artifacts"
if not (os.path.ismount(DRIVE) and os.path.isdir(f"{DRIVE}/MyDrive")):
    raise RuntimeError("Google Drive is not mounted; mount it and re-run this cell.")
os.makedirs(ARTIFACTS, exist_ok=True)
os.environ["BYTERESEARCH_ARTIFACTS"] = ARTIFACTS
os.environ["BYTERESEARCH_METRICS_DIR"] = "/content/envs"   # venvs cannot live on Drive
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("no HF_TOKEN secret; continuing without it")
print("artifacts:", ARTIFACTS)

In [ ]:
# GPU, torch, CUDA and bf16 check.
!nvidia-smi
import torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| available", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())

In [ ]:
# Clone (or update) the code and install the main environment.
REPO = "/content/embedding-research"
if os.path.isdir(f"{REPO}/.git"):
    !git -C {REPO} pull --ff-only
else:
    !git clone https://github.com/Aarushvinod/embedding-research.git {REPO}
%cd {REPO}
!pip install -q -r ByteResearch/requirements.txt

## Runner
`br(stage, *flags, models=..., conditions=..., gpus=..., pilot=..., smoke=...)` runs
`python -m ByteResearch.run <stage> ...`. It refuses to run unless Drive is mounted and the artifacts root is on
Drive. Each unit's console output is also saved to `ARTIFACTS/logs/<stage>/<run>.log`.

In [ ]:
MODELS = "byt5-small,mt5-small"   # models for evaluation and interp: all | small | base | large | byt5 | mt5 | keys
CONDITIONS = "C1,C2,C3"           # C1 = en->X, C2 = fa->X, C3 = ru->X
PILOT = False                     # interp stages on fixed row subsets

import shlex
from IPython import get_ipython


def br(stage, *flags, models=None, conditions=None, gpus=1, pilot=None, smoke=False):
    # Runs one ByteResearch stage or composite on Drive-backed artifacts and raises if it fails.
    root = os.environ.get("BYTERESEARCH_ARTIFACTS", "")
    if not (os.path.ismount(DRIVE) and root.startswith(DRIVE + "/")):
        raise RuntimeError("Google Drive is not mounted (or the artifacts root is not on it); re-run the Drive cell.")
    args = [stage, "--model", models or ("all" if smoke else MODELS),
            "--condition", conditions or ("all" if smoke else CONDITIONS), "--gpus", str(gpus)]
    if smoke:
        args.append("--smoke")
    if (PILOT if pilot is None else pilot) and stage.startswith("interp"):
        args.append("--pilot")
    cmd = f"cd {REPO} && python -m ByteResearch.run " + " ".join(shlex.quote(str(a)) for a in [*args, *flags])
    print("$", cmd)
    ip = get_ipython()
    ip.system(cmd)
    if ip.user_ns.get("_exit_code", 0):
        raise RuntimeError(f"{stage} failed (exit {ip.user_ns['_exit_code']})")

In [ ]:
# Metrics environment for COMET and MetricX (transformers<5) in /content/envs; rebuilt on each new VM.
br("setup-metrics")

In [ ]:
# Optional smoke run of the whole pipeline on tiny settings (ARTIFACTS/smoke). Skip it for real runs.
RUN_SMOKE = False
if RUN_SMOKE:
    for stage in ("data", "sweep.select", "train", "eval", "interp"):
        br(stage, smoke=True, pilot=True)

In [ ]:
# Data: FLORES, NTREX, exclusion set, GlotLID, LaBSE, sacreBLEU SPM, the FLORES-calibrated filter thresholds
# with their statistics (ARTIFACTS/data/thresholds), NLLB pairs, Wikipedia,
# the per-pair filter table (ARTIFACTS/data/nllb/FILTER_TABLE.json; up to 200k pairs per direction), then all data
# for the candidate targets Estonian, Sinhala and Tamil (pairs from every hub and Wikipedia; never trained on) with
# their yield table in ARTIFACTS/data/nllb/CANDIDATE_TABLE.json. Already-built data is skipped; FLORES, NTREX, the
# exclusion set and the thresholds rebuild themselves when the languages change, and pair files when the cap does.
MEASURE_CANDIDATES = True         # False: skip the candidate pairs and Wikipedia samples
if MEASURE_CANDIDATES:
    br("data")
else:
    for stage in ("data.flores", "data.ntrex", "data.exclusion", "data.lid_model", "data.labse_model",
                  "data.sacrebleu", "data.thresholds", "data.nllb", "data.wiki", "data.table"):
        br(stage)
# One part only: br("data.nllb", "--target", "sr", conditions="C2")
# After choosing the targets, delete the unused candidates' pairs and Wikipedia samples (permanent):
# br("data.purge", "--purge", "sq,uk")
!cd {REPO} && python -c "import json, os; from ByteResearch.corpora.thresholds import format_thresholds; print(format_thresholds(json.load(open(os.environ['BYTERESEARCH_ARTIFACTS'] + '/data/thresholds/thresholds.json', encoding='utf-8'))))"
!cd {REPO} && python -c "import json, os; from ByteResearch.corpora.nllb import format_filter_table; print(format_filter_table(json.load(open(os.environ['BYTERESEARCH_ARTIFACTS'] + '/data/nllb/FILTER_TABLE.json', encoding='utf-8'))))"
if MEASURE_CANDIDATES:
    !cd {REPO} && python -c "import json, os; from ByteResearch.corpora.nllb import format_filter_table; print(format_filter_table(json.load(open(os.environ['BYTERESEARCH_ARTIFACTS'] + '/data/nllb/CANDIDATE_TABLE.json', encoding='utf-8'))))"

In [ ]:
# Training. This cell's own selection decides what is trained.
TRAIN_MODELS = "byt5-small"       # e.g. "byt5-small", "byt5-small,mt5-small", "small" or "all"
TRAIN_CONDITIONS = "all"          # "all" or e.g. "C2" / "C1,C3"
TRAIN_LR = None                   # None: LR sweep over C1-C3 + per-model selection; or e.g. "3e-4"
TRAIN_GPUS = 1                    # >1 relaunches through torchrun; the global batch stays 64

import glob, json
if TRAIN_LR is None:
    br("sweep", models=TRAIN_MODELS, conditions="all", gpus=TRAIN_GPUS)
    br("sweep.select", models=TRAIN_MODELS, conditions="all")
    for f in sorted(glob.glob(f"{ARTIFACTS}/results/sweep/*/selection.json")):
        agg = json.load(open(f))["aggregate"]
        print(f.split("/")[-2], "lr =", agg["lr"], "| mean dev chrF++ =", round(agg["mean_chrf_pp"], 2))
br("train", *(["--lr", TRAIN_LR] if TRAIN_LR else []), models=TRAIN_MODELS, conditions=TRAIN_CONDITIONS,
   gpus=TRAIN_GPUS)
# Smaller per-device batch for a big model: br("train", "--per-device-batch", "8", models="byt5-base", gpus=2)

In [ ]:
# Translation evaluation: hypotheses, then chrF++/spBLEU/COMET/MetricX, then ByT5 vs mT5 per size x condition.
import glob, json
br("translate")
br("score")
br("compare", "--did")
for f in sorted(glob.glob(f"{ARTIFACTS}/results/compare/*__C*/flores_devtest.json")):
    agg = json.load(open(f))["aggregate"]
    for metric, c in agg["pooled"].items():
        print(f"{f.split('/')[-2]:10} {metric:8} byt5={c['byt5']:.4f} mt5={c['mt5']:.4f} "
              f"delta={c['delta']:+.4f} [{c['ci_low']:+.4f}, {c['ci_high']:+.4f}] p={c['p_value']:.4f} "
              f"better={c['better']} wins={agg['wins'][metric]}")

In [ ]:
# Hubness: states -> similarity -> ranks.
br("interp.hubness")
# br("interp.hubness.similarity")                                            # one measurement
# br("interp.hubness.states", models="byt5-small", conditions="C2", pilot=True)

In [ ]:
# DecoderLens: logprob, lid -> summary.
br("interp.decoderlens")
# br("interp.decoderlens.logprob")                                           # one measurement
# br("interp.decoderlens.lid", models="mt5-small", conditions="C3", pilot=True)

In [ ]:
# LAPE (encoder): activation_probs -> scores -> selection/overlap/correlation -> ablation -> ablation_compare.
br("interp.lape")
# br("interp.lape.overlap", "--variant", "k1")                               # one measurement, one k
# br("interp.lape.ablation", "--variant", "k1__shared", models="byt5-small", conditions="C2", pilot=True)

In [ ]:
# Steering: directions -> select -> logprob/generate/check -> metrics -> summary -> compare.
br("interp.steering")
# br("interp.steering.estimate", models="byt5-small", conditions="C2")     # timing only, writes nothing
# br("interp.steering.logprob", models="mt5-small", conditions="C2", pilot=True)

In [ ]:
# Stitching (recipients C2/C3, donors from C1-C3): logprob, generate -> metrics -> summary -> compare.
br("interp.stitching")
# br("interp.stitching.logprob", "--stitch-layers", "0,4,8")                 # one measurement, chosen layers
# br("interp.stitching.generate", models="byt5-small", conditions="C2", pilot=True)

In [ ]:
# Done/missing grid of every stage for the current parameters.
br("status")